In [ ]:
# ============================================================
# CheXpert → NIH (LOAD TRAINED ResNet50) + PubMedBERT prompts (Fusion) | ONE CELL (FIXED BACKPROP)
# - Loads pretrained CNN checkpoint: /kaggle/input/resnett/pytorch/default/1/best_resnet50.pt
# - CNN extracts image feature f(x) (2048) + logits_cnn
# - PubMedBERT encodes prompt embedding d_t (768) from CSV metadata
# - Train ONLY a small fusion head producing delta logits:
#       logits = logits_cnn + delta([f(x), d_t])
# - Saves best fusion checkpoint + NIH ROC plot + metrics CSV
# - GPU ONLY (T4)
# ============================================================

import os, re, time, math, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import roc_auc_score, precision_recall_curve, auc as sk_auc, roc_curve
from tqdm.auto import tqdm

# If transformers not installed:
# !pip -q install transformers
from transformers import AutoTokenizer, AutoModel

# -------------------------
# 0) GPU ONLY + REPRO
# -------------------------
assert torch.cuda.is_available(), "❌ CUDA not available. Kaggle: Settings → Accelerator → GPU (T4)."
DEVICE = torch.device("cuda:0")
print("✅ GPU:", torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

# ============================================================
# 1) PATHS (KAGGLE)
# ============================================================
CNN_CKPT = "/kaggle/input/resnett/pytorch/default/1/best_resnet50.pt"

NIH_IMG   = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224"
NIH_CSV   = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv"
NIH_TEST  = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt"

CHEX_BASE  = "/kaggle/input/chexpert"
CHEX_TRAIN = "/kaggle/input/chexpert/train.csv"
CHEX_VAL   = "/kaggle/input/chexpert/valid.csv"

OUT_DIR = "./output_resnet50_loaded_plus_pubmedbert"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# 2) LABELS (Unified 5)
# ============================================================
UNIFIED_LABELS = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Pleural Effusion"]

# ============================================================
# 3) HELPERS
# ============================================================
def read_txt_lines(path):
    with open(path, "r") as f:
        return [x.strip() for x in f.readlines() if x.strip()]

def parse_chex_patient_id(path_str):
    m = re.search(r"patient(\d+)", str(path_str))
    return m.group(1) if m else "unknown"

def detect_chex_image_root(chex_base, sample_rel_paths):
    candidates = [
        chex_base,
        os.path.join(chex_base, "CheXpert-v1.0-small"),
        os.path.join(chex_base, "CheXpert-v1.0-small", "CheXpert-v1.0-small"),
        os.path.join(chex_base, "CheXpert-v1.0"),
        os.path.join(chex_base, "CheXpert-v1.0", "CheXpert-v1.0-small"),
        os.path.join(chex_base, "CheXpert-v1.0", "CheXpert-v1.0"),
    ]

    def try_join(root, rel):
        rel = str(rel)
        rel2 = rel.lstrip("./")
        rel3 = rel.replace("CheXpert-v1.0-small/", "")
        rel4 = rel.replace("CheXpert-v1.0/", "")
        tries = [
            os.path.join(root, rel),
            os.path.join(root, rel2),
            os.path.join(root, rel3),
            os.path.join(root, rel4),
            os.path.join(root, "CheXpert-v1.0-small", rel),
            os.path.join(root, "CheXpert-v1.0-small", rel3),
        ]
        for p in tries:
            if os.path.exists(p):
                return p, tries
        return None, tries

    best_root, best_found = None, -1
    for root in candidates:
        found = 0
        for rel in sample_rel_paths:
            p, _ = try_join(root, rel)
            if p is not None:
                found += 1
        if found > best_found:
            best_found = found
            best_root = root

    if best_found <= 0:
        rel0 = sample_rel_paths[0]
        _, tries = try_join(chex_base, rel0)
        raise RuntimeError(
            f"❌ Could not locate CheXpert images.\nchex_base={chex_base}\nsample rel={rel0}\nTried:\n  - " +
            "\n  - ".join(tries[:12])
        )

    print(f"✅ Detected CheXpert image root: {best_root} (matched {best_found}/{len(sample_rel_paths)} samples)")

    def resolver(rel):
        p, tries = try_join(best_root, rel)
        if p is None:
            raise FileNotFoundError(
                f"❌ Image not found.\nroot={best_root}\nrel={rel}\nTried:\n  - " + "\n  - ".join(tries[:12])
            )
        return p
    return resolver

def build_nih_df(split_list_txt, nih_csv_path):
    wanted_imgs = set(read_txt_lines(split_list_txt))
    df = pd.read_csv(nih_csv_path)

    if "Image Index" not in df.columns or "Finding Labels" not in df.columns:
        raise RuntimeError("❌ NIH CSV missing required columns: 'Image Index' and/or 'Finding Labels'.")

    df = df[df["Image Index"].isin(wanted_imgs)].copy()
    if len(df) == 0:
        raise RuntimeError(f"❌ NIH split is empty after filtering: {split_list_txt}")

    findings = df["Finding Labels"].fillna("")

    def has(lbl):
        return findings.apply(lambda s: int(lbl in str(s).split("|")))

    df["Atelectasis"] = has("Atelectasis")
    df["Cardiomegaly"] = has("Cardiomegaly")
    df["Consolidation"] = has("Consolidation")
    df["Edema"] = has("Edema")
    df["Pleural Effusion"] = has("Effusion")

    pid_col = "Patient ID" if "Patient ID" in df.columns else None
    df["patient_id"] = df[pid_col].astype(str) if pid_col else "unknown"

    df["age_meta"]  = df["Patient Age"].astype(str) if "Patient Age" in df.columns else "unknown"
    df["sex_meta"]  = df["Patient Gender"].astype(str) if "Patient Gender" in df.columns else "unknown"
    df["view_meta"] = df["View Position"].astype(str) if "View Position" in df.columns else "unknown"
    df["ap_pa_meta"] = "unknown"

    df = df.rename(columns={"Image Index":"image"})
    df = df[["image","patient_id","age_meta","sex_meta","view_meta","ap_pa_meta"] + UNIFIED_LABELS].reset_index(drop=True)
    return df

def build_chex_df(csv_path):
    df = pd.read_csv(csv_path)

    img_col = None
    for c in ["Path","path","image_path","Image"]:
        if c in df.columns:
            img_col = c
            break
    if img_col is None:
        raise RuntimeError(f"❌ CheXpert CSV missing image path column. Columns={list(df.columns)[:30]}")

    pleural_col = None
    for c in ["Pleural Effusion","Pleural_Effusion"]:
        if c in df.columns:
            pleural_col = c
            break
    needed = ["Atelectasis","Cardiomegaly","Consolidation","Edema"]
    for c in needed:
        if c not in df.columns:
            raise RuntimeError(f"❌ CheXpert CSV missing label column: {c}")
    if pleural_col is None:
        raise RuntimeError("❌ CheXpert CSV missing Pleural Effusion column.")

    frontal_col = None
    for c in ["Frontal/Lateral","Frontal_Lateral"]:
        if c in df.columns:
            frontal_col = c
            df = df[df[c].astype(str).str.lower().str.contains("frontal")].copy()
            break

    df = df.rename(columns={img_col:"image", pleural_col:"Pleural Effusion"}).copy()

    for c in UNIFIED_LABELS:
        df[c] = df[c].replace(-1, 0).fillna(0).astype(np.float32)

    df["patient_id"] = df["image"].apply(parse_chex_patient_id).astype(str)

    df["age_meta"] = df["Age"].astype(str) if "Age" in df.columns else "unknown"
    df["sex_meta"] = df["Sex"].astype(str) if "Sex" in df.columns else "unknown"
    df["view_meta"] = df[frontal_col].astype(str) if frontal_col else "unknown"
    df["ap_pa_meta"] = df["AP/PA"].astype(str) if "AP/PA" in df.columns else "unknown"

    df = df[["image","patient_id","age_meta","sex_meta","view_meta","ap_pa_meta"] + UNIFIED_LABELS].reset_index(drop=True)
    return df

def compute_pos_weights(labels_np):
    pos = labels_np.sum(axis=0)
    neg = labels_np.shape[0] - pos
    return torch.tensor(neg / (pos + 1e-6), dtype=torch.float32)

def make_prompt(dataset_name, age, sex, view, ap_pa):
    return (
        f"Dataset: {dataset_name}. "
        f"Patient age: {age}. Sex: {sex}. View: {view}. Projection: {ap_pa}. "
        f"Describe potential acquisition differences or imaging artifacts visible in this CXR. "
        f"Estimate exposure quality, structural clarity, and anatomical consistency."
    )

# ============================================================
# 4) DATASET
# ============================================================
class CXRDataset(Dataset):
    def __init__(self, df, img_resolver, dataset_name, transform=None):
        self.df = df.reset_index(drop=True)
        self.img_resolver = img_resolver
        self.transform = transform
        self.dataset_name = dataset_name
        self.images = self.df["image"].tolist()
        self.labels = self.df[UNIFIED_LABELS].values.astype(np.float32)

        self.prompts = []
        for _, r in self.df.iterrows():
            self.prompts.append(make_prompt(
                dataset_name,
                str(r.get("age_meta", "unknown")),
                str(r.get("sex_meta", "unknown")),
                str(r.get("view_meta", "unknown")),
                str(r.get("ap_pa_meta", "unknown")),
            ))

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        rel = self.images[idx]
        img_path = self.img_resolver(rel)
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        y = torch.from_numpy(self.labels[idx])
        return img, y, self.prompts[idx]

def collate_fn(batch):
    xs, ys, texts = zip(*batch)
    return torch.stack(xs, 0), torch.stack(ys, 0), list(texts)

# ============================================================
# 5) TRANSFORMS
# ============================================================
train_tf = T.Compose([
    T.RandomResizedCrop(224, scale=(0.85, 1.0)),
    T.RandomRotation(7),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

eval_tf = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# ============================================================
# 6) LOAD DATA
# ============================================================
print("\n📦 Loading manifests...")
chex_train_df = build_chex_df(CHEX_TRAIN)
chex_val_df   = build_chex_df(CHEX_VAL)
nih_test_df   = build_nih_df(NIH_TEST, NIH_CSV)
print(f"CheX Train: {len(chex_train_df):,} | CheX Val: {len(chex_val_df):,} | NIH Test: {len(nih_test_df):,}")

print("\n🔧 Detecting CheXpert image root...")
chex_resolver = detect_chex_image_root(CHEX_BASE, chex_train_df["image"].head(80).tolist())

def nih_resolver(rel):
    p = os.path.join(NIH_IMG, str(rel))
    if not os.path.exists(p):
        raise FileNotFoundError(f"❌ NIH image not found: {p}")
    return p

# ============================================================
# 7) LOADERS
# ============================================================
BATCH_SIZE = 32
EPOCHS = 8
LR = 2e-4
WEIGHT_DECAY = 0.05
WARMUP_FRAC = 0.05
MAX_TEXT_LEN = 96

train_ds = CXRDataset(chex_train_df, chex_resolver, "CheXpert", transform=train_tf)
val_ds   = CXRDataset(chex_val_df,   chex_resolver, "CheXpert", transform=eval_tf)
test_ds  = CXRDataset(nih_test_df,   nih_resolver,  "NIH",      transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0, pin_memory=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True, collate_fn=collate_fn)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True, collate_fn=collate_fn)

# ============================================================
# 8) CNN: load trained checkpoint + feature/logit split
# ============================================================
assert os.path.exists(CNN_CKPT), f"❌ CNN checkpoint not found: {CNN_CKPT}"

class ResNet50FeatLogit(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        m = models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        self.m = m

    def forward(self, x):
        m = self.m
        x = m.conv1(x); x = m.bn1(x); x = m.relu(x); x = m.maxpool(x)
        x = m.layer1(x); x = m.layer2(x); x = m.layer3(x); x = m.layer4(x)
        x = m.avgpool(x)
        feat = torch.flatten(x, 1)     # (B,2048)
        logits = m.fc(feat)            # (B,5)
        return feat, logits

cnn = ResNet50FeatLogit(num_classes=len(UNIFIED_LABELS)).to(DEVICE)

ck = torch.load(CNN_CKPT, map_location="cpu")
if isinstance(ck, dict) and "model_state_dict" in ck:
    state = ck["model_state_dict"]
elif isinstance(ck, dict) and "state_dict" in ck:
    state = ck["state_dict"]
elif isinstance(ck, dict):
    state = ck
else:
    state = ck

missing, unexpected = cnn.load_state_dict(state, strict=False)
print("✅ Loaded CNN.")
if missing: print("  missing keys (sample):", missing[:6])
if unexpected: print("  unexpected keys (sample):", unexpected[:6])

# Freeze CNN
cnn.eval()
for p in cnn.parameters():
    p.requires_grad = False

# ============================================================
# 9) PubMedBERT / BiomedBERT (frozen)
# ============================================================
TEXT_MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
print("\n🧠 Loading BiomedBERT:", TEXT_MODEL_NAME)
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_encoder = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(DEVICE)

text_encoder.eval()
for p in text_encoder.parameters():
    p.requires_grad = False

def mean_pool(last_hidden, attention_mask):
    mask = attention_mask.unsqueeze(-1).type_as(last_hidden)
    summed = (last_hidden * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1e-6)
    return summed / denom

# ============================================================
# 10) Fusion head (TRAINABLE)
# ============================================================
class FusionDeltaHead(nn.Module):
    def __init__(self, in_img=2048, in_txt=768, num_classes=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_img + in_txt, 1024),
            nn.ReLU(inplace=True),
            nn.Dropout(0.25),
            nn.Linear(1024, num_classes),
        )
    def forward(self, img_feat, txt_feat):
        return self.net(torch.cat([img_feat, txt_feat], dim=1))

fusion = FusionDeltaHead(num_classes=len(UNIFIED_LABELS)).to(DEVICE)

# sanity: fusion must require grad
assert any(p.requires_grad for p in fusion.parameters()), "❌ Fusion head has no trainable params!"

# Loss
pos_w = compute_pos_weights(train_ds.labels).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

optimizer = torch.optim.AdamW(fusion.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

total_steps = EPOCHS * len(train_loader)
warmup_steps = max(1, int(WARMUP_FRAC * total_steps))
def lr_lambda(step):
    if step < warmup_steps:
        return step / warmup_steps
    prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * prog))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

use_amp = True
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

# ============================================================
# 11) Forward helpers (TRAIN vs EVAL)
# ============================================================
def encode_text(texts):
    tok = tokenizer(texts, padding=True, truncation=True, max_length=MAX_TEXT_LEN, return_tensors="pt")
    input_ids = tok["input_ids"].to(DEVICE, non_blocking=True)
    attn = tok["attention_mask"].to(DEVICE, non_blocking=True)
    out = text_encoder(input_ids=input_ids, attention_mask=attn)
    txt_feat = mean_pool(out.last_hidden_state, attn)
    return txt_feat

def forward_batch_train(x, texts):
    # grads must flow into fusion
    x = x.to(DEVICE, non_blocking=True)
    with torch.no_grad():
        img_feat, logits_cnn = cnn(x)          # frozen
    with torch.no_grad():
        txt_feat = encode_text(texts)          # frozen
    delta = fusion(img_feat, txt_feat)         # trainable
    return logits_cnn + delta                  # has grad_fn via delta

@torch.no_grad()
def forward_batch_eval(x, texts):
    x = x.to(DEVICE, non_blocking=True)
    img_feat, logits_cnn = cnn(x)
    txt_feat = encode_text(texts)
    delta = fusion(img_feat, txt_feat)
    return logits_cnn + delta

# ============================================================
# 12) Metrics + plots
# ============================================================
@torch.no_grad()
def evaluate(loader, name):
    fusion.eval()
    all_p, all_y = [], []
    for x, y, texts in tqdm(loader, desc=f"Eval {name}", leave=False):
        logits = forward_batch_eval(x, texts)
        p = torch.sigmoid(logits).float().cpu().numpy()
        all_p.append(p)
        all_y.append(y.numpy())

    P = np.vstack(all_p)
    Y = np.vstack(all_y)

    aucs, pr_aucs = [], []
    for i, lbl in enumerate(UNIFIED_LABELS):
        try:
            aucs.append(roc_auc_score(Y[:, i], P[:, i]))
        except ValueError:
            aucs.append(np.nan)
        prec, rec, _ = precision_recall_curve(Y[:, i], P[:, i])
        pr_aucs.append(sk_auc(rec, prec))
    return {
        "P": P, "Y": Y,
        "aucs": aucs, "pr_aucs": pr_aucs,
        "macro_auc": float(np.nanmean(aucs)),
        "macro_pr": float(np.nanmean(pr_aucs)),
    }

def plot_roc(P, Y, title, save_path):
    plt.figure(figsize=(9,7))
    for i, lbl in enumerate(UNIFIED_LABELS):
        try:
            fpr, tpr, _ = roc_curve(Y[:, i], P[:, i])
            plt.plot(fpr, tpr, label=lbl)
        except ValueError:
            pass
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

# ============================================================
# 13) Train fusion head
# ============================================================
best_val_auc = -1
best_path = os.path.join(OUT_DIR, "best_fusion_delta.pt")
last_path = os.path.join(OUT_DIR, "last_fusion_delta.pt")

print("\n🚀 Training fusion head (CNN frozen, BiomedBERT frozen)")

for epoch in range(1, EPOCHS + 1):
    fusion.train()
    t0 = time.time()
    run_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Train {epoch}/{EPOCHS}")
    for x, y, texts in pbar:
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = forward_batch_train(x, texts)
            loss = criterion(logits, y)

        # ✅ now loss has grad_fn (because fusion is in graph)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        run_loss += loss.item() * y.size(0)
        pbar.set_postfix(loss=float(loss.item()), lr=float(optimizer.param_groups[0]["lr"]))

    run_loss /= len(train_loader.dataset)
    val_out = evaluate(val_loader, "CheX Val")
    val_auc = val_out["macro_auc"]
    print(f"Epoch {epoch:02d} | train_loss={run_loss:.4f} | val_macro_auc={val_auc:.4f} | time={(time.time()-t0)/60:.1f} min")

    torch.save({
        "epoch": epoch,
        "fusion_state_dict": fusion.state_dict(),
        "cnn_ckpt": CNN_CKPT,
        "text_model": TEXT_MODEL_NAME,
        "labels": UNIFIED_LABELS,
        "best_val_auc_so_far": best_val_auc,
        "note": "logits = logits_cnn + fusion_delta([cnn_feat, pubmedbert_prompt_emb])"
    }, last_path)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save({
            "epoch": epoch,
            "fusion_state_dict": fusion.state_dict(),
            "cnn_ckpt": CNN_CKPT,
            "text_model": TEXT_MODEL_NAME,
            "labels": UNIFIED_LABELS,
            "best_val_auc": best_val_auc,
            "note": "logits = logits_cnn + fusion_delta([cnn_feat, pubmedbert_prompt_emb])"
        }, best_path)
        print(f"  ✅ saved BEST -> {best_path}")

print(f"\n✅ Best CheX Val macro AUC = {best_val_auc:.4f}")
print(f"📌 Best fusion checkpoint: {best_path}")

# ============================================================
# 14) NIH TEST (combined)
# ============================================================
print("\n🧪 NIH TEST (Combined CNN + PubMedBERT)")
ck = torch.load(best_path, map_location="cpu")
fusion.load_state_dict(ck["fusion_state_dict"])
fusion.eval()

test_out = evaluate(test_loader, "NIH Test")
print("\n================ NIH TEST RESULTS (COMBINED) ================")
print(f"Macro AUC:    {test_out['macro_auc']:.4f}")
print(f"Macro PR-AUC: {test_out['macro_pr']:.4f}\n")
for lbl, a, p in zip(UNIFIED_LABELS, test_out["aucs"], test_out["pr_aucs"]):
    print(f"{lbl:18s} | AUC={a:.4f} | PR-AUC={p:.4f}")

roc_png = os.path.join(OUT_DIR, "roc_nih_test_combined.png")
plot_roc(test_out["P"], test_out["Y"], "ROC (CheXpert→NIH) | Loaded ResNet50 + PubMedBERT fusion", roc_png)
print(f"\n📌 ROC saved: {roc_png}")

metrics_csv = os.path.join(OUT_DIR, "nih_test_metrics_combined.csv")
pd.DataFrame({"label": UNIFIED_LABELS, "auc_roc": test_out["aucs"], "pr_auc": test_out["pr_aucs"]}).to_csv(metrics_csv, index=False)
print(f"📌 Metrics saved: {metrics_csv}")

print("\n✅ DONE.")


✅ GPU: Tesla T4

📦 Loading manifests...
CheX Train: 191,027 | CheX Val: 202 | NIH Test: 25,596

🔧 Detecting CheXpert image root...
✅ Detected CheXpert image root: /kaggle/input/chexpert (matched 80/80 samples)
✅ Loaded CNN.
  missing keys (sample): ['m.conv1.weight', 'm.bn1.weight', 'm.bn1.bias', 'm.bn1.running_mean', 'm.bn1.running_var', 'm.layer1.0.conv1.weight']
  unexpected keys (sample): ['net.conv1.weight', 'net.bn1.weight', 'net.bn1.bias', 'net.bn1.running_mean', 'net.bn1.running_var', 'net.bn1.num_batches_tracked']

🧠 Loading BiomedBERT: microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext

🚀 Training fusion head (CNN frozen, BiomedBERT frozen)


Train 1/8:   0%|          | 0/5970 [00:00<?, ?it/s]

In [ ]:
# ============================================================
# OPTION A (Journal-quality):
# 1) Load TRAINED CNN (ResNet50) from .pt  -> FREEZE (no CNN training)
# 2) Precompute image artifact/quality features from pixels (brightness/contrast/sharpness/entropy/edges/clipping)
# 3) Precompute CNN probs (from .pt) for each sample
# 4) Train PubMedBERT ONLY on enriched prompts (CSV metadata + artifacts + CNN probs)
# 5) NIH test: CNN-only vs Text-only vs Combined(avg probs)
# 6) Save:
#    - best_pubmedbert_text.pt
#    - metrics CSV
#    - ROC plots
#    - RAG tables (documents + summary) + feature hist plots
# ============================================================

import os, re, time, math, random, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models

from sklearn.metrics import roc_auc_score, precision_recall_curve, auc as sk_auc, roc_curve
from tqdm.auto import tqdm

# transformers
# !pip -q install transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# ============================================================
# 0) GPU ONLY
# ============================================================
assert torch.cuda.is_available(), "❌ CUDA not available. Kaggle: Settings → Accelerator → GPU (T4)."
DEVICE = torch.device("cuda:0")
print("✅ GPU:", torch.cuda.get_device_name(0))

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

# ============================================================
# 1) PATHS
# ============================================================
CNN_CKPT = "/kaggle/input/resnett/pytorch/default/1/best_resnet50.pt"

NIH_IMG   = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/images-224/images-224"
NIH_CSV   = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/Data_Entry_2017.csv"
NIH_TEST  = "/kaggle/input/nih-chest-x-ray-14-224x224-resized/test_list_NIH.txt"

CHEX_BASE  = "/kaggle/input/chexpert"
CHEX_TRAIN = "/kaggle/input/chexpert/train.csv"
CHEX_VAL   = "/kaggle/input/chexpert/valid.csv"

OUT_DIR = "./output_optionA_cnnFrozen_pubmedbert"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# 2) CONFIG
# ============================================================
UNIFIED_LABELS = ["Atelectasis","Cardiomegaly","Consolidation","Edema","Pleural Effusion"]

# Feature extraction can be heavy on full CheXpert. Use caps.
MAX_TRAIN_SAMPLES = 30000   # set None for full train (very slow)
MAX_VAL_SAMPLES   = 8000    # set None for full val
MAX_TEST_SAMPLES  = None    # NIH test can be full

BATCH_IMG_FEAT = 64         # for CNN-prob extraction
BATCH_TEXT_TRAIN = 16
BATCH_TEXT_EVAL  = 32

EPOCHS = 8
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP_FRAC = 0.05
MAX_TEXT_LEN = 160

# train only last N layers + head (faster on T4). set 12 for full finetune
UNFREEZE_LAST_N_LAYERS = 4

# ============================================================
# 3) HELPERS: IO + DF builders
# ============================================================
def read_txt_lines(path):
    with open(path, "r") as f:
        return [x.strip() for x in f.readlines() if x.strip()]

def parse_chex_patient_id(path_str):
    m = re.search(r"patient(\d+)", str(path_str))
    return m.group(1) if m else "unknown"

def detect_chex_image_root(chex_base, sample_rel_paths):
    candidates = [
        chex_base,
        os.path.join(chex_base, "CheXpert-v1.0-small"),
        os.path.join(chex_base, "CheXpert-v1.0-small", "CheXpert-v1.0-small"),
        os.path.join(chex_base, "CheXpert-v1.0"),
        os.path.join(chex_base, "CheXpert-v1.0", "CheXpert-v1.0-small"),
        os.path.join(chex_base, "CheXpert-v1.0", "CheXpert-v1.0"),
    ]
    def try_join(root, rel):
        rel = str(rel)
        rel2 = rel.lstrip("./")
        rel3 = rel.replace("CheXpert-v1.0-small/", "")
        rel4 = rel.replace("CheXpert-v1.0/", "")
        tries = [
            os.path.join(root, rel),
            os.path.join(root, rel2),
            os.path.join(root, rel3),
            os.path.join(root, rel4),
            os.path.join(root, "CheXpert-v1.0-small", rel),
            os.path.join(root, "CheXpert-v1.0-small", rel3),
        ]
        for p in tries:
            if os.path.exists(p):
                return p, tries
        return None, tries

    best_root, best_found = None, -1
    for root in candidates:
        found = 0
        for rel in sample_rel_paths:
            p, _ = try_join(root, rel)
            found += (p is not None)
        if found > best_found:
            best_found = found
            best_root = root

    if best_found <= 0:
        rel0 = sample_rel_paths[0]
        _, tries = try_join(chex_base, rel0)
        raise RuntimeError("❌ Could not locate CheXpert images.\nTried:\n  - " + "\n  - ".join(tries[:12]))

    print(f"✅ Detected CheXpert image root: {best_root} (matched {best_found}/{len(sample_rel_paths)} samples)")

    def resolver(rel):
        p, tries = try_join(best_root, rel)
        if p is None:
            raise FileNotFoundError("❌ Image not found.\nTried:\n  - " + "\n  - ".join(tries[:12]))
        return p
    return resolver

def build_nih_df(split_list_txt, nih_csv_path):
    wanted_imgs = set(read_txt_lines(split_list_txt))
    df = pd.read_csv(nih_csv_path)

    if "Image Index" not in df.columns or "Finding Labels" not in df.columns:
        raise RuntimeError("❌ NIH CSV missing columns: Image Index / Finding Labels")

    df = df[df["Image Index"].isin(wanted_imgs)].copy()
    if len(df) == 0:
        raise RuntimeError("❌ NIH split empty after filtering list")

    findings = df["Finding Labels"].fillna("")
    def has(lbl): return findings.apply(lambda s: int(lbl in str(s).split("|")))

    df["Atelectasis"] = has("Atelectasis")
    df["Cardiomegaly"] = has("Cardiomegaly")
    df["Consolidation"] = has("Consolidation")
    df["Edema"] = has("Edema")
    df["Pleural Effusion"] = has("Effusion")

    df["patient_id"] = df["Patient ID"].astype(str) if "Patient ID" in df.columns else "unknown"
    df["age_meta"]  = df["Patient Age"].astype(str) if "Patient Age" in df.columns else "unknown"
    df["sex_meta"]  = df["Patient Gender"].astype(str) if "Patient Gender" in df.columns else "unknown"
    df["view_meta"] = df["View Position"].astype(str) if "View Position" in df.columns else "unknown"
    df["ap_pa_meta"] = "unknown"

    df = df.rename(columns={"Image Index":"image"})
    keep = ["image","patient_id","age_meta","sex_meta","view_meta","ap_pa_meta"] + UNIFIED_LABELS
    return df[keep].reset_index(drop=True)

def build_chex_df(csv_path):
    df = pd.read_csv(csv_path)

    img_col = None
    for c in ["Path","path","image_path","Image"]:
        if c in df.columns: img_col = c; break
    if img_col is None:
        raise RuntimeError("❌ CheXpert CSV missing Path/path.")

    pleural_col = None
    for c in ["Pleural Effusion","Pleural_Effusion"]:
        if c in df.columns: pleural_col = c; break
    if pleural_col is None:
        raise RuntimeError("❌ CheXpert CSV missing Pleural Effusion column.")

    frontal_col = None
    for c in ["Frontal/Lateral","Frontal_Lateral"]:
        if c in df.columns:
            frontal_col = c
            df = df[df[c].astype(str).str.lower().str.contains("frontal")].copy()
            break

    df = df.rename(columns={img_col:"image", pleural_col:"Pleural Effusion"}).copy()

    for c in UNIFIED_LABELS:
        df[c] = df[c].replace(-1, 0).fillna(0).astype(np.float32)

    df["patient_id"] = df["image"].apply(parse_chex_patient_id).astype(str)
    df["age_meta"] = df["Age"].astype(str) if "Age" in df.columns else "unknown"
    df["sex_meta"] = df["Sex"].astype(str) if "Sex" in df.columns else "unknown"
    df["view_meta"] = df[frontal_col].astype(str) if frontal_col else "unknown"
    df["ap_pa_meta"] = df["AP/PA"].astype(str) if "AP/PA" in df.columns else "unknown"

    keep = ["image","patient_id","age_meta","sex_meta","view_meta","ap_pa_meta"] + UNIFIED_LABELS
    return df[keep].reset_index(drop=True)

def compute_pos_weights(labels_np):
    pos = labels_np.sum(axis=0)
    neg = labels_np.shape[0] - pos
    return torch.tensor(neg / (pos + 1e-6), dtype=torch.float32)

# ============================================================
# 4) IMAGE ARTIFACT/QUALITY FEATURES (Option A)
# ============================================================
def _entropy_from_uint8(gray_uint8):
    hist = np.bincount(gray_uint8.ravel(), minlength=256).astype(np.float64)
    p = hist / (hist.sum() + 1e-12)
    p = p[p > 0]
    return float(-(p * np.log2(p)).sum())

def _laplacian_var(gray_f):
    # gray_f: float32 in [0,1], shape (H,W)
    k = np.array([[0,1,0],[1,-4,1],[0,1,0]], dtype=np.float32)
    # conv valid-ish
    g = gray_f
    lap = (
        k[0,1]*g[:-2,1:-1] + k[1,0]*g[1:-1,:-2] + k[1,1]*g[1:-1,1:-1] + k[1,2]*g[1:-1,2:] + k[2,1]*g[2:,1:-1]
    )
    return float(lap.var())

def _edge_density(gray_f):
    # simple Sobel magnitude thresholding
    g = gray_f
    gx = (g[1:-1,2:] - g[1:-1,:-2]) * 0.5
    gy = (g[2:,1:-1] - g[:-2,1:-1]) * 0.5
    mag = np.sqrt(gx*gx + gy*gy)
    thr = np.percentile(mag, 90)  # adaptive threshold
    return float((mag > thr).mean())

def artifact_features(pil_img):
    # returns dict of artifact proxies
    gray = pil_img.convert("L")
    arr_u8 = np.array(gray, dtype=np.uint8)
    arr = arr_u8.astype(np.float32) / 255.0

    mean = float(arr.mean())
    std = float(arr.std())
    ent = _entropy_from_uint8(arr_u8)
    sharp = _laplacian_var(arr)
    edge = _edge_density(arr)

    clip_low = float((arr <= 0.01).mean())
    clip_high = float((arr >= 0.99).mean())

    return {
        "q_mean": mean,
        "q_std": std,
        "q_entropy": ent,
        "q_sharp_lapvar": sharp,
        "q_edge_density": edge,
        "q_clip_low": clip_low,
        "q_clip_high": clip_high,
    }

# ============================================================
# 5) PROMPT BUILDER (CSV meta + artifacts + CNN probs)
# ============================================================
def make_prompt(dataset_name, age, sex, view, ap_pa, q, cnn_probs):
    # q: dict artifact feats, cnn_probs: list length=5
    # round to keep prompt compact
    def r(x): 
        try: return f"{float(x):.4f}"
        except: return "unknown"

    probs_str = ", ".join([f"{UNIFIED_LABELS[i]}={cnn_probs[i]:.3f}" for i in range(len(UNIFIED_LABELS))])

    return (
        f"Dataset={dataset_name}; Age={age}; Sex={sex}; View={view}; Projection={ap_pa}. "
        f"Quality: mean={r(q['q_mean'])}, std={r(q['q_std'])}, entropy={r(q['q_entropy'])}, "
        f"sharpness_lapvar={r(q['q_sharp_lapvar'])}, edge_density={r(q['q_edge_density'])}, "
        f"clip_low={r(q['q_clip_low'])}, clip_high={r(q['q_clip_high'])}. "
        f"CNN_probs: {probs_str}. "
        f"Task: infer pathology likelihood while accounting for acquisition differences and imaging artifacts "
        f"(exposure, motion blur, noise, rotation, structural clarity)."
    )

# ============================================================
# 6) LOAD DATAFRAMES + RESOLVERS
# ============================================================
print("\n📦 Loading manifests...")
chex_train_df = build_chex_df(CHEX_TRAIN)
chex_val_df   = build_chex_df(CHEX_VAL)
nih_test_df   = build_nih_df(NIH_TEST, NIH_CSV)

print(f"CheX Train: {len(chex_train_df):,} | CheX Val: {len(chex_val_df):,} | NIH Test: {len(nih_test_df):,}")

print("\n🔧 Detecting CheXpert image root...")
chex_resolver = detect_chex_image_root(CHEX_BASE, chex_train_df["image"].head(120).tolist())

def nih_resolver(rel):
    p = os.path.join(NIH_IMG, str(rel))
    if not os.path.exists(p):
        raise FileNotFoundError(f"❌ NIH image not found: {p}")
    return p

# Apply sample caps (for speed)
def cap_df(df, max_n, name):
    if max_n is None or len(df) <= max_n:
        return df
    df2 = df.sample(max_n, random_state=SEED).reset_index(drop=True)
    print(f"⚠ Using subset for {name}: {len(df2):,}/{len(df):,}")
    return df2

chex_train_df_cap = cap_df(chex_train_df, MAX_TRAIN_SAMPLES, "CheX Train (feature+text training)")
chex_val_df_cap   = cap_df(chex_val_df,   MAX_VAL_SAMPLES,   "CheX Val (feature+text validation)")
nih_test_df_cap   = cap_df(nih_test_df,   MAX_TEST_SAMPLES,  "NIH Test (evaluation)")

# ============================================================
# 7) LOAD CNN (ResNet50) + LOAD .pt properly + FREEZE
# ============================================================
assert os.path.exists(CNN_CKPT), f"❌ CNN checkpoint not found: {CNN_CKPT}"

class ResNetMultiLabel(nn.Module):
    def __init__(self, num_classes=5):
        super().__init__()
        m = models.resnet50(weights=None)
        m.fc = nn.Linear(m.fc.in_features, num_classes)
        self.net = m  # keep "net" so net.* matches
    def forward(self, x):
        return self.net(x)

cnn = ResNetMultiLabel(num_classes=len(UNIFIED_LABELS)).to(DEVICE)

def normalize_state_dict(sd):
    new = {}
    for k,v in sd.items():
        if k.startswith("module."):
            k = k[len("module."):]
        if k.startswith("m."):
            k = "net." + k[len("m."):]
        elif not k.startswith("net."):
            k = "net." + k
        new[k] = v
    return new

ck = torch.load(CNN_CKPT, map_location="cpu")
if isinstance(ck, dict) and "state_dict" in ck:
    state = ck["state_dict"]
elif isinstance(ck, dict) and "model_state_dict" in ck:
    state = ck["model_state_dict"]
elif isinstance(ck, dict):
    state = ck
else:
    state = ck

state = normalize_state_dict(state)
missing, unexpected = cnn.load_state_dict(state, strict=False)
print("\n✅ Loaded CNN checkpoint")
print("   missing keys:", len(missing))
print("   unexpected keys:", len(unexpected))

cnn.eval()
for p in cnn.parameters():
    p.requires_grad = False

# ============================================================
# 8) TRANSFORMS (for CNN inference)
# ============================================================
eval_tf = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

# ============================================================
# 9) FEATURE EXTRACTION DATASET (artifact feats + cnn probs)
# ============================================================
class FeatExtractDataset(Dataset):
    def __init__(self, df, resolver, dataset_name):
        self.df = df.reset_index(drop=True)
        self.resolver = resolver
        self.dataset_name = dataset_name
        self.labels = self.df[UNIFIED_LABELS].values.astype(np.float32)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        r = self.df.iloc[idx]
        rel = r["image"]
        img_path = self.resolver(rel)

        pil = Image.open(img_path).convert("RGB")
        q = artifact_features(pil)  # Option A
        x = eval_tf(pil)

        meta = {
            "dataset": self.dataset_name,
            "image": str(rel),
            "patient_id": str(r["patient_id"]),
            "age_meta": str(r["age_meta"]),
            "sex_meta": str(r["sex_meta"]),
            "view_meta": str(r["view_meta"]),
            "ap_pa_meta": str(r["ap_pa_meta"]),
            **q
        }
        y = torch.from_numpy(self.labels[idx])
        return x, y, meta

def feat_collate(batch):
    xs, ys, metas = zip(*batch)
    return torch.stack(xs,0), torch.stack(ys,0), list(metas)

@torch.no_grad()
def precompute_features(df, resolver, dataset_name, out_csv):
    if os.path.exists(out_csv):
        print(f"✅ Using cached features: {out_csv}")
        return pd.read_csv(out_csv)

    ds = FeatExtractDataset(df, resolver, dataset_name)
    loader = DataLoader(ds, batch_size=BATCH_IMG_FEAT, shuffle=False, num_workers=0, pin_memory=True, collate_fn=feat_collate)

    rows = []
    for x, y, metas in tqdm(loader, desc=f"Precompute {dataset_name} feats", leave=False):
        x = x.to(DEVICE, non_blocking=True)
        logits = cnn(x)                         # (B,5)
        probs = torch.sigmoid(logits).float().cpu().numpy()
        y_np = y.numpy()

        for i in range(len(metas)):
            row = dict(metas[i])
            # labels
            for j,lbl in enumerate(UNIFIED_LABELS):
                row[f"y_{lbl}"] = float(y_np[i, j])
                row[f"cnn_{lbl}"] = float(probs[i, j])
            rows.append(row)

    feat_df = pd.DataFrame(rows)
    feat_df.to_csv(out_csv, index=False)
    print(f"📌 Saved features: {out_csv}")
    return feat_df

# Precompute (or load cached)
train_feat_csv = os.path.join(OUT_DIR, "chex_train_features.csv")
val_feat_csv   = os.path.join(OUT_DIR, "chex_val_features.csv")
nih_feat_csv   = os.path.join(OUT_DIR, "nih_test_features.csv")

train_feat = precompute_features(chex_train_df_cap, chex_resolver, "CheXpert", train_feat_csv)
val_feat   = precompute_features(chex_val_df_cap,   chex_resolver, "CheXpert", val_feat_csv)
nih_feat   = precompute_features(nih_test_df_cap,   nih_resolver,  "NIH",      nih_feat_csv)

# ============================================================
# 10) RAG TABLES (documents + summary + feature plots)
# ============================================================
def save_rag_tables(feat_df, name_prefix):
    # build prompt_text column
    prompts = []
    for _, r in feat_df.iterrows():
        q = {k: r[k] for k in ["q_mean","q_std","q_entropy","q_sharp_lapvar","q_edge_density","q_clip_low","q_clip_high"]}
        cnn_probs = [r[f"cnn_{lbl}"] for lbl in UNIFIED_LABELS]
        prompts.append(make_prompt(
            str(r["dataset"]),
            str(r["age_meta"]), str(r["sex_meta"]), str(r["view_meta"]), str(r["ap_pa_meta"]),
            q, cnn_probs
        ))
    feat_df = feat_df.copy()
    feat_df["prompt_text"] = prompts

    # documents table
    doc_cols = ["dataset","image","patient_id","age_meta","sex_meta","view_meta","ap_pa_meta",
                "q_mean","q_std","q_entropy","q_sharp_lapvar","q_edge_density","q_clip_low","q_clip_high"]
    for lbl in UNIFIED_LABELS:
        doc_cols += [f"y_{lbl}", f"cnn_{lbl}"]
    doc_cols += ["prompt_text"]

    docs_path = os.path.join(OUT_DIR, f"rag_{name_prefix}_documents.csv")
    feat_df[doc_cols].to_csv(docs_path, index=False)

    # summary table
    summary_rows = []
    summary_rows.append({"metric":"n_samples", "value": int(len(feat_df))})
    summary_rows.append({"metric":"n_patients", "value": int(feat_df["patient_id"].nunique())})

    for k in ["q_mean","q_std","q_entropy","q_sharp_lapvar","q_edge_density","q_clip_low","q_clip_high"]:
        summary_rows.append({"metric":f"{k}_mean", "value": float(feat_df[k].mean())})
        summary_rows.append({"metric":f"{k}_std",  "value": float(feat_df[k].std())})

    for lbl in UNIFIED_LABELS:
        summary_rows.append({"metric":f"prevalence_{lbl}", "value": float(feat_df[f"y_{lbl}"].mean())})

    summ_path = os.path.join(OUT_DIR, f"rag_{name_prefix}_summary.csv")
    pd.DataFrame(summary_rows).to_csv(summ_path, index=False)

    # feature hist plots (journal-friendly)
    for k in ["q_mean","q_sharp_lapvar","q_entropy","q_edge_density"]:
        plt.figure(figsize=(7,5))
        plt.hist(feat_df[k].values, bins=60)
        plt.title(f"{name_prefix}: Distribution of {k}")
        plt.xlabel(k); plt.ylabel("count")
        plt.grid(True, alpha=0.3)
        p = os.path.join(OUT_DIR, f"{name_prefix}_hist_{k}.png")
        plt.savefig(p, dpi=300, bbox_inches="tight")
        plt.close()

    print(f"\n📌 RAG docs: {docs_path}")
    print(f"📌 RAG summary: {summ_path}")
    print(f"📌 Feature hist plots saved in: {OUT_DIR}")
    return feat_df

train_feat = save_rag_tables(train_feat, "chex_train")
val_feat   = save_rag_tables(val_feat,   "chex_val")
nih_feat   = save_rag_tables(nih_feat,   "nih_test")

# ============================================================
# 11) TEXT DATASET for PubMedBERT TRAINING (uses prompt_text)
# ============================================================
class PromptDataset(Dataset):
    def __init__(self, feat_df):
        self.df = feat_df.reset_index(drop=True)
        self.texts = self.df["prompt_text"].tolist()
        self.Y = self.df[[f"y_{lbl}" for lbl in UNIFIED_LABELS]].values.astype(np.float32)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        return self.texts[idx], torch.from_numpy(self.Y[idx])

def text_collate(batch):
    texts, ys = zip(*batch)
    return list(texts), torch.stack(ys, 0)

train_text_ds = PromptDataset(train_feat)
val_text_ds   = PromptDataset(val_feat)

train_text_loader = DataLoader(train_text_ds, batch_size=BATCH_TEXT_TRAIN, shuffle=True,  num_workers=0, pin_memory=True, collate_fn=text_collate)
val_text_loader   = DataLoader(val_text_ds,   batch_size=BATCH_TEXT_EVAL,  shuffle=False, num_workers=0, pin_memory=True, collate_fn=text_collate)

# ============================================================
# 12) LOAD + TRAIN PubMedBERT ONLY
# ============================================================
TEXT_MODEL_NAME = "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext"
print("\n🧠 Loading PubMedBERT/BiomedBERT:", TEXT_MODEL_NAME)

tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_model = AutoModelForSequenceClassification.from_pretrained(
    TEXT_MODEL_NAME,
    num_labels=len(UNIFIED_LABELS),
    problem_type="multi_label_classification"
).to(DEVICE)

# freeze all, then unfreeze last N layers + classifier
for p in text_model.parameters():
    p.requires_grad = False

for p in text_model.classifier.parameters():
    p.requires_grad = True

base = getattr(text_model, text_model.base_model_prefix)  # bert
enc_layers = base.encoder.layer
nL = len(enc_layers)
for i in range(nL - UNFREEZE_LAST_N_LAYERS, nL):
    for p in enc_layers[i].parameters():
        p.requires_grad = True

trainable = sum(p.numel() for p in text_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in text_model.parameters())
print(f"✅ Text trainable params: {trainable/1e6:.2f}M / {total/1e6:.2f}M (last {UNFREEZE_LAST_N_LAYERS} layers + head)")

pos_w = compute_pos_weights(train_feat[[f"y_{lbl}" for lbl in UNIFIED_LABELS]].values.astype(np.float32)).to(DEVICE)
criterion = nn.BCEWithLogitsLoss(pos_weight=pos_w)

optimizer = torch.optim.AdamW([p for p in text_model.parameters() if p.requires_grad], lr=LR, weight_decay=WEIGHT_DECAY)

total_steps = EPOCHS * len(train_text_loader)
warmup_steps = max(1, int(WARMUP_FRAC * total_steps))
def lr_lambda(step):
    if step < warmup_steps:
        return step / warmup_steps
    prog = (step - warmup_steps) / max(1, total_steps - warmup_steps)
    return 0.5 * (1.0 + math.cos(math.pi * prog))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

use_amp = True
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

@torch.no_grad()
def eval_text(loader, name):
    text_model.eval()
    all_p, all_y = [], []
    for texts, y in tqdm(loader, desc=f"EvalText {name}", leave=False):
        tok = tokenizer(texts, padding=True, truncation=True, max_length=MAX_TEXT_LEN, return_tensors="pt")
        input_ids = tok["input_ids"].to(DEVICE)
        attn = tok["attention_mask"].to(DEVICE)
        logits = text_model(input_ids=input_ids, attention_mask=attn).logits
        p = torch.sigmoid(logits).float().cpu().numpy()
        all_p.append(p)
        all_y.append(y.numpy())
    P = np.vstack(all_p); Y = np.vstack(all_y)

    aucs, pr_aucs = [], []
    for i, lbl in enumerate(UNIFIED_LABELS):
        try:
            aucs.append(roc_auc_score(Y[:, i], P[:, i]))
        except ValueError:
            aucs.append(np.nan)
        prec, rec, _ = precision_recall_curve(Y[:, i], P[:, i])
        pr_aucs.append(sk_auc(rec, prec))
    return P, Y, aucs, pr_aucs, float(np.nanmean(aucs)), float(np.nanmean(pr_aucs))

best_val_auc = -1
best_text_path = os.path.join(OUT_DIR, "best_pubmedbert_text.pt")

print("\n🚀 Training PubMedBERT ONLY (CNN frozen; artifacts + CNN probs injected as text)")
for epoch in range(1, EPOCHS + 1):
    text_model.train()
    epoch_loss = 0.0
    t0 = time.time()

    pbar = tqdm(train_text_loader, desc=f"TextTrain {epoch}/{EPOCHS}")
    for texts, y in pbar:
        y = y.to(DEVICE)

        tok = tokenizer(texts, padding=True, truncation=True, max_length=MAX_TEXT_LEN, return_tensors="pt")
        input_ids = tok["input_ids"].to(DEVICE)
        attn = tok["attention_mask"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = text_model(input_ids=input_ids, attention_mask=attn).logits
            loss = criterion(logits, y)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        epoch_loss += loss.item() * y.size(0)
        pbar.set_postfix(loss=float(loss.item()), lr=float(optimizer.param_groups[0]["lr"]))

    epoch_loss /= len(train_text_loader.dataset)

    _, _, _, _, val_auc, _ = eval_text(val_text_loader, "CheX Val")
    print(f"Epoch {epoch:02d} | loss={epoch_loss:.4f} | val_macro_auc={val_auc:.4f} | time={(time.time()-t0)/60:.1f} min")

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        torch.save({
            "epoch": epoch,
            "state_dict": text_model.state_dict(),
            "labels": UNIFIED_LABELS,
            "best_val_auc": best_val_auc,
            "text_model_name": TEXT_MODEL_NAME,
            "unfreeze_last_n_layers": UNFREEZE_LAST_N_LAYERS,
        }, best_text_path)
        print(f"  ✅ saved BEST -> {best_text_path}")

print(f"\n✅ Best CheX Val macro AUC (Text-only) = {best_val_auc:.4f}")
print(f"📌 Best text checkpoint: {best_text_path}")

# ============================================================
# 13) NIH TEST: CNN-only vs Text-only vs Combined
# ============================================================
ck = torch.load(best_text_path, map_location="cpu")
text_model.load_state_dict(ck["state_dict"])
text_model.eval()

# NIH prompts already in nih_feat["prompt_text"] (from RAG stage)
nih_text_ds = PromptDataset(nih_feat)
nih_text_loader = DataLoader(nih_text_ds, batch_size=BATCH_TEXT_EVAL, shuffle=False, num_workers=0, pin_memory=True, collate_fn=text_collate)

@torch.no_grad()
def eval_text_probs(loader, name):
    text_model.eval()
    all_p, all_y = [], []
    for texts, y in tqdm(loader, desc=f"EvalText {name}", leave=False):
        tok = tokenizer(texts, padding=True, truncation=True, max_length=MAX_TEXT_LEN, return_tensors="pt")
        input_ids = tok["input_ids"].to(DEVICE)
        attn = tok["attention_mask"].to(DEVICE)
        logits = text_model(input_ids=input_ids, attention_mask=attn).logits
        p = torch.sigmoid(logits).float().cpu().numpy()
        all_p.append(p)
        all_y.append(y.numpy())
    return np.vstack(all_p), np.vstack(all_y)

P_txt, Y = eval_text_probs(nih_text_loader, "NIH")

# CNN-only probabilities from precomputed columns
P_cnn = nih_feat[[f"cnn_{lbl}" for lbl in UNIFIED_LABELS]].values.astype(np.float32)
P_comb = 0.5 * (P_cnn + P_txt)

def perclass_metrics(Y, P):
    aucs, pr_aucs = [], []
    for i,lbl in enumerate(UNIFIED_LABELS):
        try:
            aucs.append(roc_auc_score(Y[:, i], P[:, i]))
        except ValueError:
            aucs.append(np.nan)
        prec, rec, _ = precision_recall_curve(Y[:, i], P[:, i])
        pr_aucs.append(sk_auc(rec, prec))
    return aucs, pr_aucs, float(np.nanmean(aucs)), float(np.nanmean(pr_aucs))

def plot_roc(Y, P, title, save_path):
    plt.figure(figsize=(9,7))
    for i,lbl in enumerate(UNIFIED_LABELS):
        try:
            fpr, tpr, _ = roc_curve(Y[:, i], P[:, i])
            plt.plot(fpr, tpr, label=lbl)
        except ValueError:
            pass
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title(title)
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.close()

au_cnn, pr_cnn, mauc_cnn, mpr_cnn = perclass_metrics(Y, P_cnn)
au_txt, pr_txt, mauc_txt, mpr_txt = perclass_metrics(Y, P_txt)
au_cmb, pr_cmb, mauc_cmb, mpr_cmb = perclass_metrics(Y, P_comb)

print("\n================ NIH TEST RESULTS ================")
print(f"CNN-only     | Macro AUC={mauc_cnn:.4f} | Macro PR={mpr_cnn:.4f}")
print(f"PubMedBERT   | Macro AUC={mauc_txt:.4f} | Macro PR={mpr_txt:.4f}")
print(f"COMBINED(avg)| Macro AUC={mauc_cmb:.4f} | Macro PR={mpr_cmb:.4f}")

print("\nPer-class (COMBINED):")
for lbl, a, p in zip(UNIFIED_LABELS, au_cmb, pr_cmb):
    print(f"{lbl:18s} | AUC={a:.4f} | PR-AUC={p:.4f}")

roc1 = os.path.join(OUT_DIR, "roc_nih_cnn_only.png")
roc2 = os.path.join(OUT_DIR, "roc_nih_text_only.png")
roc3 = os.path.join(OUT_DIR, "roc_nih_combined_avg.png")
plot_roc(Y, P_cnn, "ROC NIH | CNN-only (loaded .pt, frozen)", roc1)
plot_roc(Y, P_txt, "ROC NIH | PubMedBERT (trained on CSV+artifacts+cnn probs)", roc2)
plot_roc(Y, P_comb, "ROC NIH | Combined avg probs", roc3)

print(f"\n📌 ROC plots saved:\n - {roc1}\n - {roc2}\n - {roc3}")

# Metrics CSV (journal table)
rows = []
for i,lbl in enumerate(UNIFIED_LABELS):
    rows.append({
        "label": lbl,
        "auc_cnn": au_cnn[i], "pr_cnn": pr_cnn[i],
        "auc_text": au_txt[i], "pr_text": pr_txt[i],
        "auc_combined": au_cmb[i], "pr_combined": pr_cmb[i],
    })
rows.append({
    "label": "MACRO",
    "auc_cnn": mauc_cnn, "pr_cnn": mpr_cnn,
    "auc_text": mauc_txt, "pr_text": mpr_txt,
    "auc_combined": mauc_cmb, "pr_combined": mpr_cmb,
})
metrics_csv = os.path.join(OUT_DIR, "nih_metrics_cnn_text_combined.csv")
pd.DataFrame(rows).to_csv(metrics_csv, index=False)
print(f"📌 Metrics saved: {metrics_csv}")

print("\n✅ DONE (CNN not trained). Option A artifact features + CNN probs used to train PubMedBERT only.")


✅ GPU: Tesla T4

📦 Loading manifests...
CheX Train: 191,027 | CheX Val: 202 | NIH Test: 25,596

🔧 Detecting CheXpert image root...
✅ Detected CheXpert image root: /kaggle/input/chexpert (matched 120/120 samples)
⚠ Using subset for CheX Train (feature+text training): 30,000/191,027

✅ Loaded CNN checkpoint
   missing keys: 0
   unexpected keys: 0


Precompute CheXpert feats:   0%|          | 0/469 [00:00<?, ?it/s]

KeyboardInterrupt: 